Data Preprocessing & Augmentation (SMOTE)

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import joblib

print("Loading and preprocessing dataset...")
df = pd.read_csv('diabetes.csv')

# Handle biologically impossible zeros
columns_with_zeros = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
for col in columns_with_zeros:
    df[col] = df[col].replace(0, np.nan)
    df[col] = df[col].fillna(df[col].median())
    
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

# Standardization
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
joblib.dump(scaler, 'scaler.pkl')

# Data Augmentation (SMOTE)
print(f"Original training class distribution:\n{y_train.value_counts()}")
smote = SMOTE(random_state=42)
X_train_aug, y_train_aug = smote.fit_resample(X_train_scaled, y_train)
print(f"\nAugmented training class distribution:\n{y_train_aug.value_counts()}")

Loading and preprocessing dataset...
Original training class distribution:
Outcome
0    400
1    214
Name: count, dtype: int64

Augmented training class distribution:
Outcome
0    400
1    400
Name: count, dtype: int64


Model Training & Evaluation

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import joblib

models = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=500),
    "Random Forest": RandomForestClassifier(random_state=42, n_estimators=100)
}

best_model = None
best_auc = 0
best_model_name = ""

for name, model in models.items():
    print(f"\n--- Training {name} ---")
    cv_scores = cross_val_score(model, X_train_aug, y_train_aug, cv=5, scoring='accuracy')
    print(f"CV Accuracy (5-Fold): {np.mean(cv_scores):.4f}")
    
    model.fit(X_train_aug, y_train_aug)
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    
    auc = roc_auc_score(y_test, y_prob)
    print(f"Test Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(f"F1-Score: {f1_score(y_test, y_pred):.4f}")
    print(f"AUC-ROC: {auc:.4f}")
    
    if auc > best_auc:
        best_auc = auc
        best_model = model
        best_model_name = name

joblib.dump(best_model, 'diapredict_best_model.pkl')
print(f"\nSuccessfully serialized '{best_model_name}' to disk.")


--- Training Logistic Regression ---
CV Accuracy (5-Fold): 0.7387
Test Accuracy: 0.7143
F1-Score: 0.6207
AUC-ROC: 0.8115

--- Training Random Forest ---
CV Accuracy (5-Fold): 0.8225
Test Accuracy: 0.7532
F1-Score: 0.6724
AUC-ROC: 0.8142

Successfully serialized 'Random Forest' to disk.


Database Initialization

In [3]:
import sqlite3

print("Initializing SQLite Database...")
conn = sqlite3.connect('diapredict.db')
cursor = conn.cursor()

cursor.execute('''
    CREATE TABLE IF NOT EXISTS users (
        user_id INTEGER PRIMARY KEY AUTOINCREMENT,
        email TEXT UNIQUE NOT NULL,
        password TEXT NOT NULL,
        name TEXT NOT NULL
    )
''')

cursor.execute('''
    CREATE TABLE IF NOT EXISTS assessments (
        assessment_id INTEGER PRIMARY KEY AUTOINCREMENT,
        user_id INTEGER,
        assessment_date TEXT,
        pregnancies INTEGER, glucose REAL, blood_pressure REAL,
        skin_thickness REAL, insulin REAL, bmi REAL,
        dpf REAL, age INTEGER, risk_level TEXT, probability REAL,
        FOREIGN KEY(user_id) REFERENCES users(user_id)
    )
''')
conn.commit()
conn.close()
print("Database 'diapredict.db' created successfully.")

Initializing SQLite Database...
Database 'diapredict.db' created successfully.
